# 第 13 课 · 从目标函数到正确的梯度

完成前 12 课后，你已经见过参数更新。本课要求你能解释每个张量为何存在，并通过反例发现实现错误。使用 CPU 即可，不依赖前课权重。

先回答：`loss` 数值相同，梯度一定相同吗？提示 token 是否该计算 SFT 损失？GRPO 的 old policy 和 reference policy 是否相同？先写下判断，运行后再修改。

## 1 · SFT 的标签为何要错开一位

因果语言模型计算 $p_\theta(x_t\mid x_{<t})$。给定提示 $q$ 和回答 $a$，本课使用

$$L_{SFT}=-\frac{1}{N_a}\sum_{t\in\text{answer+EOS}}\log p_\theta(x_t\mid x_{<t}).$$

logits 形状是 `[batch, sequence, vocabulary]`。位置 t 的 logits 预测位置 t+1 的标签；提示与 padding 的标签设成 `-100`，EOS 仍参与学习。Hugging Face 因果模型内部通常已经做 shift，外部再 shift 一次会错位。

下面用独立张量检验对齐，之后才把同一规则交给模型。


In [ ]:
import json, math
from pathlib import Path
import torch
import torch.nn.functional as F
torch.manual_seed(13)
ids = torch.tensor([[1, 2, 3, 4, 0]])  # 两个提示 token、答案、EOS、padding
labels = torch.tensor([[-100, -100, 3, 4, -100]])
logits = torch.randn(1, 5, 6, requires_grad=True)
loss = F.cross_entropy(logits[:, :-1].reshape(-1, 6), labels[:, 1:].reshape(-1), ignore_index=-100)
manual = -(logits.log_softmax(-1)[0, 1, 3] + logits.log_softmax(-1)[0, 2, 4])/2
assert torch.allclose(loss, manual)
loss.backward()
assert logits.grad[0, [0, 3, 4]].abs().sum() == 0
print('PASS: 只有预测答案与 EOS 的两个位置收到梯度。', float(loss.detach()))


## 2 · 策略梯度：奖励不是可微标签

策略目标 $J(\theta)=\mathbb E_{a\sim\pi_\theta}[r(a)]$ 的梯度为
$\mathbb E[r(a)\nabla_\theta\log\pi_\theta(a)]$，奖励函数本身无需可微。用采样到的回答近似这个期望，基线可以降低方差。

本格枚举四个动作，比较精确期望梯度与一个具有相同梯度的 surrogate loss。概率权重必须 `detach()`：采样发生时的分布不应在这条估计式里被再次微分。这不是“所有概率都要 detach”，而是要区分采样权重和被优化的 log probability。


In [ ]:
scores = torch.tensor([.2, -.1, .4, -.2], dtype=torch.float64, requires_grad=True)
rewards = torch.tensor([0., 1., 0., 0.], dtype=torch.float64)
probs = scores.softmax(-1)
exact = -(probs * rewards).sum()
g_exact, = torch.autograd.grad(exact, scores, retain_graph=True)
surrogate = -(probs.detach() * rewards * probs.log()).sum()
g_surrogate, = torch.autograd.grad(surrogate, scores, retain_graph=True)
wrong = -(probs * rewards * probs.log()).sum()
g_wrong, = torch.autograd.grad(wrong, scores)
assert torch.allclose(g_exact, g_surrogate)
assert not torch.allclose(g_exact, g_wrong)
print('精确梯度:', g_exact.tolist())
print('遗漏 detach 的梯度:', g_wrong.tolist())


## 3 · GRPO：组内比较与裁剪

本教学定义 $A_i=(r_i-\bar r)/(\sigma_r+10^{-6})$，标准差使用总体版本 `unbiased=False`。不同库的标准差、序列/批次归一化和优势尺度可能不同，迁移代码时必须核对实现。

设 $\rho_t=\exp(\log\pi_\theta-\log\pi_{old})$，策略损失为

$$L=-\operatorname{mean}_{\text{有效回答 token}}\min(\rho_t A,\operatorname{clip}(\rho_t,1-\epsilon,1+\epsilon)A).$$

old policy 是采样该批回答的策略快照；reference policy 是提供 KL 约束的参考模型。它们用途不同。old log-prob 与优势应固定，new log-prob 要保留梯度。对负优势，裁剪的有效方向与正优势不同，不能直接把 ratio 裁剪后相乘代替 `min`。

下面测试全对组、全错组、混合组、正负优势和 padding。它是张量单元实验，不是新的 verl trainer。


In [ ]:
reward_groups = torch.tensor([[0., 0., 0., 0.], [1., 1., 1., 1.], [0., 1., 0., 1.]])
advantages = (reward_groups-reward_groups.mean(-1, keepdim=True))/(reward_groups.std(-1, keepdim=True, unbiased=False)+1e-6)
assert advantages[:2].abs().sum() == 0
assert advantages[2, 1] > 0 and advantages[2, 0] < 0
ratios = torch.tensor([[1.4, 1.4, 1.0], [.6, .6, 1.0]], requires_grad=True)
adv = torch.tensor([[1.], [-1.]])
mask = torch.tensor([[1., 1., 0.], [1., 1., 0.]])
terms = torch.minimum(ratios*adv, ratios.clamp(.8, 1.2)*adv)
clipped_loss = -(terms*mask).sum()/mask.sum()
clipped_loss.backward()
assert ratios.grad.abs().sum() == 0  # 两行都越过了有利方向的裁剪边界
print('优势:', advantages)
print('PASS: 全同奖励无相对信号；有利方向超限后梯度被裁剪。')
# 不能把 seq-mean 与 token-mean 混为同一个目标。
values = torch.tensor([[1., 0., 0.], [3., 3., 3.]])
m = torch.tensor([[1., 0., 0.], [1., 1., 1.]])
token_mean = (values*m).sum()/m.sum()
sequence_mean = ((values*m).sum(-1)/m.sum(-1)).mean()
print('token mean:', float(token_mean), 'sequence mean:', float(sequence_mean))
assert token_mean != sequence_mean


## 4 · KL 估计与代码审查

记 $d=\log\pi_{ref}-\log\pi_\theta$。常用采样项 $e^d-d-1$ 在数值上非负。当样本确实来自当前策略且支持集满足条件时，其期望对应 $D_{KL}(\pi_\theta\|\pi_{ref})$；用旧策略样本或有限批次时，不能直接当作精确 KL。

此处验证离散分布上的期望恒等式。实际序列训练还需要回答掩码、EOS 处理与相应的采样/更新约定。


In [ ]:
p = torch.tensor([.1, .2, .7], dtype=torch.float64)
q = torch.tensor([.3, .3, .4], dtype=torch.float64)
d = q.log()-p.log()
k3 = d.exp()-d-1
exact_kl = (p*(p.log()-q.log())).sum()
assert torch.allclose((p*k3).sum(), exact_kl)
out = Path('support/experiment13_output'); out.mkdir(exist_ok=True)
(out/'checks.json').write_text(json.dumps(dict(shift=True, policy_gradient=True,
    clipping=True, masks=True, kl_identity=True, exact_kl=float(exact_kl)), indent=2))
print('PASS: KL 期望一致。请继续完成下面的代码审查题。')


## 5 · 验收：你能否发现“跑得动”的错误

1. 把第 1 格的 label mask 全部取消。哪些 token 改变了训练目标？
2. 把混合组换成一个正确、七个错误。比较组大小变化后的优势尺度。
3. 在裁剪格中把 ratio 改为正优势 0.6、负优势 1.4。先预测梯度是否为零，再运行。
4. 从第 03 课找到 old log-prob、reference log-prob 和 EOS mask。用具体变量解释它们的生命周期。

交付一页审查记录：写出张量形状、被优化的目标、三个可能静默出错的位置。仅通过断言不足以证明完整算法正确。

参考：[DeepSeekMath](https://arxiv.org/abs/2402.03300)、[PPO](https://arxiv.org/abs/1707.06347)、[PyTorch autograd](https://pytorch.org/docs/stable/autograd.html)。下一课把理论用于修复冷启动。
